# L15: 长期记忆实现

> 让 Agent 跨会话「记住」知识和对话历史

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional
from datetime import datetime

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.memory.long_term import LongTermMemory
from backend.llm.models import Message
print("✓ 模块导入成功")

## 15.1 什么是长期记忆？

### 定义

**长期记忆 (Long-term Memory)** = 持久化的、可跨会话访问的知识存储。

### 长期记忆 vs 短期记忆

| 特性 | 短期记忆 | 长期记忆 |
|------|----------|----------|
| **存储内容** | 当前对话消息 | 向量化知识库 |
| **生命周期** | 会话结束即清空 | 持久存储 |
| **检索方式** | 顺序获取全部 | 语义相似度检索 |
| **容量限制** | 受 Context Window 限制 | 几乎无限制 |
| **用途** | 维持对话上下文 | 知识复用、历史查询 |

### 长期记忆的典型应用

```
场景 1: 跨会话记忆
  用户（第一次对话）: "我叫小明"
  用户（一个月后）: "我叫什么名字？"
  Agent: "你的名字是小明。" ← 从长期记忆中检索

场景 2: 知识库问答
  用户: "公司的请假流程是什么？"
  Agent: 从员工手册（已存入向量库）检索并回答
```

## 15.2 查看项目中的长期记忆实现

In [ ]:
import inspect
from backend.memory.long_term import LongTermMemory

print("=== LongTermMemory 源码 ===\n")
print(inspect.getsource(LongTermMemory))

## 15.3 长期记忆的基本操作

In [ ]:
async def demo_long_term_memory():
    """演示长期记忆的基本操作"""
    
    # 创建长期记忆实例
    memory = LongTermMemory(
        collection_name="demo",
        persist_directory="./chroma_db_demo"
    )
    
    print("=== 1. 添加记忆 ===\n")
    
    # 添加不同类型的记忆
    memories = [
        {
            "content": "用户的名字叫小明，是一名软件工程师",
            "metadata": {"type": "user_info", "user_id": "user_123"}
        },
        {
            "content": "用户喜欢编程语言是 Python，正在学习 Agent 开发",
            "metadata": {"type": "preference", "user_id": "user_123"}
        },
        {
            "content": "项目使用 DeepSeek 作为 LLM 提供商",
            "metadata": {"type": "project_info"}
        },
    ]
    
    for mem in memories:
        await memory.add(
            content=mem["content"],
            metadata=mem["metadata"]
        )
        print(f"✓ 添加: {mem['content'][:40]}...")
    
    print("\n=== 2. 检索记忆 ===\n")
    
    # 检索相关记忆
    queries = [
        "小明是什么工作？",
        "用户喜欢什么编程语言？",
        "项目使用什么 LLM？"
    ]
    
    for query in queries:
        results = await memory.search(query, top_k=2)
        print(f"查询: {query}")
        for i, result in enumerate(results, 1):
            print(f"  {i}. {result['content']}")
            print(f"     相似度: {result['score']:.4f}")
        print()
    
    print("=== 3. 按元数据过滤 ===\n")
    
    # 按元数据过滤检索
    results = await memory.search(
        query="用户",
        top_k=5,
        filter={"type": "user_info"}
    )
    print("用户信息:")
    for result in results:
        print(f"  - {result['content']}")
    
    # 清理演示数据
    import shutil
    if os.path.exists("./chroma_db_demo"):
        shutil.rmtree("./chroma_db_demo")

# await demo_long_term_memory()  # 取消注释以运行（需要 chromadb）

## 15.4 长期记忆的设计模式

### 模式 1: 对话历史存储

每次对话后，将重要信息存入长期记忆。

In [ ]:
class ConversationMemoryStorage:
    """
    对话记忆存储器
    将对话中的关键信息提取并存储到长期记忆
    """
    
    def __init__(self, long_term_memory: LongTermMemory):
        self.memory = long_term_memory
    
    async def store_conversation(
        self, 
        messages: List[Message],
        user_id: str = None
    ):
        """
        存储对话内容到长期记忆
        """
        for msg in messages:
            # 只存储可能包含重要信息的消息
            if self._is_important(msg):
                await self.memory.add(
                    content=msg.content,
                    metadata={
                        "role": msg.role,
                        "user_id": user_id,
                        "timestamp": datetime.now().isoformat()
                    }
                )
    
    def _is_important(self, message: Message) -> bool:
        """
        判断消息是否重要（简化版）
        """
        # 关键词判断
        important_keywords = [
            "记住", "重要", "喜欢", "名字", "叫",
            "目标是", "想要", "希望"
        ]
        
        return any(kw in message.content for kw in important_keywords)

# 使用示例
async def demo_conversation_storage():
    print("=== 对话存储示例 ===\n")
    
    # 模拟对话
    conversation = [
        Message(role="user", content="你好，我叫小明"),
        Message(role="assistant", content="你好小明！"),
        Message(role="user", content="记住，我喜欢 Python"),
        Message(role="assistant", content="好的，我记住了你喜欢 Python。"),
        Message(role="user", content="今天天气怎么样"),  # 不重要
    ]
    
    for msg in conversation:
        is_important = ConversationMemoryStorage._is_important(None, msg)
        status = "✓ 重要" if is_important else "✗ 普通"
        print(f"{status} [{msg.role}] {msg.content}")

await demo_conversation_storage()

### 模式 2: 用户画像构建

从对话中提取用户特征，构建动态用户画像。

In [ ]:
from typing import Dict, Set

class UserProfileBuilder:
    """
    用户画像构建器
    从对话中提取用户特征
    """
    
    def __init__(self):
        self.profiles: Dict[str, Dict[str, Any]] = {}
    
    async def update_from_conversation(
        self,
        user_id: str,
        messages: List[Message]
    ):
        """
        从对话中更新用户画像
        """
        if user_id not in self.profiles:
            self.profiles[user_id] = {
                "name": None,
                "interests": set(),
                "preferences": {},
                "conversation_count": 0
            }
        
        profile = self.profiles[user_id]
        profile["conversation_count"] += 1
        
        for msg in messages:
            if msg.role == "user":
                self._extract_name(profile, msg.content)
                self._extract_interests(profile, msg.content)
    
    def _extract_name(self, profile: Dict, content: str):
        """提取用户姓名"""
        import re
        # 简单模式匹配
        patterns = [
            r"我叫(\w+)",
            r"我是(\w+)",
            r"名字是(\w+)"
        ]
        for pattern in patterns:
            match = re.search(pattern, content)
            if match:
                profile["name"] = match.group(1)
    
    def _extract_interests(self, profile: Dict, content: str):
        """提取用户兴趣"""
        interest_keywords = [
            "喜欢", "爱", "感兴趣", "爱好"
        ]
        
        for kw in interest_keywords:
            if kw in content:
                # 简单提取：取关键词后的内容
                idx = content.find(kw)
                potential = content[idx + len(kw):].strip()
                if potential:
                    profile["interests"].add(potential[:10])
    
    def get_profile(self, user_id: str) -> Dict[str, Any]:
        """获取用户画像"""
        if user_id not in self.profiles:
            return {}
        
        profile = self.profiles[user_id].copy()
        profile["interests"] = list(profile["interests"])  # 转为列表
        return profile

# 测试用户画像
async def test_user_profile():
    builder = UserProfileBuilder()
    
    print("=== 用户画像构建测试 ===\n")
    
    # 模拟多次对话
    conversations = [
        [Message(role="user", content="你好，我叫小明")],
        [Message(role="user", content="我喜欢 Python 编程")],
        [Message(role="user", content="我对 AI 很感兴趣")],
        [Message(role="user", content="我还喜欢打篮球")],
    ]
    
    for conv in conversations:
        await builder.update_from_conversation("user_123", conv)
        print(f"更新: {conv[0].content}")
    
    print("\n最终用户画像:")
    print(json.dumps(builder.get_profile("user_123"), indent=2, ensure_ascii=False))

await test_user_profile()

### 模式 3: 知识库增强 (RAG)

将文档知识存入长期记忆，实现知识问答。

In [ ]:
class KnowledgeBase:
    """
    知识库管理器
    将文档分块后存入向量库
    """
    
    def __init__(self, long_term_memory: LongTermMemory):
        self.memory = long_term_memory
    
    async def add_document(
        self,
        content: str,
        doc_id: str = None,
        chunk_size: int = 500,
        chunk_overlap: int = 50
    ) -> List[str]:
        """
        添加文档到知识库
        返回添加的 chunk IDs
        """
        chunks = self._split_text(content, chunk_size, chunk_overlap)
        chunk_ids = []
        
        for i, chunk in enumerate(chunks):
            chunk_id = await self.memory.add(
                content=chunk,
                metadata={
                    "doc_id": doc_id,
                    "chunk_index": i,
                    "total_chunks": len(chunks)
                }
            )
            chunk_ids.append(chunk_id)
        
        return chunk_ids
    
    def _split_text(
        self,
        text: str,
        chunk_size: int,
        chunk_overlap: int
    ) -> List[str]:
        """
        将文本分成重叠的块
        """
        chunks = []
        start = 0
        
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            chunks.append(chunk)
            start = end - chunk_overlap
        
        return chunks
    
    async def search(
        self,
        query: str,
        top_k: int = 3,
        doc_id: str = None
    ) -> List[Dict[str, Any]]:
        """
        在知识库中搜索
        """
        filter_dict = {"doc_id": doc_id} if doc_id else None
        return await self.memory.search(
            query=query,
            top_k=top_k,
            filter=filter_dict
        )

# 测试知识库
async def test_knowledge_base():
    print("=== 知识库测试 ===\n")
    
    # 示例文档
    document = """
    Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年创建。
    它以简洁、易读的语法著称，广泛应用于 Web 开发、数据科学、人工智能等领域。
    Python 的设计哲学强调代码的可读性和简洁的语法。
    """
    
    # 模拟分块
    kb = KnowledgeBase(long_term_memory=None)  # 不实际存储
    chunks = kb._split_text(document.strip(), chunk_size=100, chunk_overlap=20)
    
    print(f"文档分块 ({len(chunks)} 个块):\n")
    for i, chunk in enumerate(chunks, 1):
        print(f"块 {i}: {chunk}\n")

await test_knowledge_base()

## 15.5 记忆管理策略

### 何时存储？

| 触发条件 | 存储内容 | 优先级 |
|----------|----------|--------|
| 对话结束 | 完整对话摘要 | 高 |
| 检测到重要信息 | 用户偏好、关键事实 | 高 |
| 定时任务 | 批量处理旧对话 | 中 |
| 用户显式请求 | 指定内容 | 高 |

### 何时检索？

| 触发条件 | 检索范围 |
|----------|----------|
| 新对话开始 | 用户画像、历史偏好 |
| 用户提问 | 相关知识库 |
| 遇到歧义 | 历史上下文 |

### 如何更新？

- **增量更新**：新信息追加
- **覆盖更新**：同类信息替换旧的
- **合并更新**：综合多条信息
- **过期清理**：删除陈旧信息

## 15.6 常见面试问题

**Q: 长期记忆和数据库有什么区别？**
- A:
  - 长期记忆使用**向量相似度**检索，数据库使用**精确匹配**
  - 长期记忆支持**模糊语义查询**，数据库需要精确的查询条件
  - 长期记忆更适合**非结构化文本**，数据库适合结构化数据
  - 实际应用中通常**结合使用**：数据库存精确信息，向量库存语义知识

**Q: 如何决定哪些信息应该存入长期记忆？**
- A:
  1. **用户明确信息**：姓名、偏好、设定
  2. **重要事实**：目标、要求、承诺
  3. **对话摘要**：每轮对话的总结
  4. **领域知识**：文档、手册、FAQ
  5. 使用 LLM 判断重要性，或关键词检测

**Q: 如何处理长期记忆中的过时信息？**
- A:
  1. **时间戳过滤**：只检索最近的信息
  2. **版本控制**：同类信息保留最新版本
  3. **定期清理**：删除超过时效的数据
  4. **置信度评分**：降低陈旧信息的权重
  5. **用户反馈**：根据用户纠正更新记忆

**Q: 向量检索的 top_k 应该设置多少？**
- A:
  - **太少**（1-2）：可能遗漏相关信息
  - **太多**（10+）：引入噪音，增加 token 消耗
  - **推荐值**：3-5，根据具体场景调整
  - 可以设置**阈值过滤**：只返回相似度高于某值的结果

**Q: RAG 中的长期记忆如何保证准确性？**
- A:
  1. **高质量文档**：确保源文档准确
  2. **合理分块**：保持语义完整性
  3. **混合检索**：向量检索 + 关键词检索
  4. **重排序**：对检索结果重新评分排序
  5. **引用标注**：让 LLM 标注信息来源
  6. **人工审核**：重要答案人工验证

---

## 总结

本课程学习了长期记忆的完整实现：

| 知识点 | 说明 |
|--------|------|
| **长期记忆** | 持久化的、可跨会话的知识存储 |
| **存储模式** | 对话历史、用户画像、知识库 |
| **检索策略** | 语义相似度 + 元数据过滤 |
| **管理策略** | 何时存储、何时检索、如何更新 |
| **RAG 集成** | 知识库作为 LLM 的外部知识源 |
| **准确性保证** | 高质量文档、合理分块、混合检索 |

**下一步**: L16 将学习 ReAct Agent 模式！